In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

In [ ]:
# torch.manual_seed(1)

In [3]:
rnn_layer = nn.RNN(input_size=5, hidden_size=2, num_layers=1, batch_first=True)

w_xh = rnn_layer.weight_ih_l0
w_hh = rnn_layer.weight_hh_l0
b_xh = rnn_layer.bias_ih_l0
b_hh = rnn_layer.bias_hh_l0

print("W_xh shape:", w_xh.shape)
print("W_hh shape:", w_hh.shape)
print("b_xh shape:", b_xh.shape)
print("b_hh shape:", b_hh.shape)

W_xh shape: torch.Size([2, 5])
W_hh shape: torch.Size([2, 2])
b_xh shape: torch.Size([2])
b_hh shape: torch.Size([2])


In [4]:
x_seq = torch.tensor([[1.0] * 5, [2.0] * 5, [3.0] * 5]).float()

# Output of simple RNN
output, hn = rnn_layer(torch.reshape(x_seq, (1, 3, 5)))

# Manually computing the output:
out_man = []
for t in range(3):
    xt = torch.reshape(x_seq[t], (1, 5))
    print(f"Time step {t} =>        ")
    print("     Input : ", xt.numpy())

    ht = torch.matmul(xt, torch.transpose(w_xh, 0, 1)) + b_hh
    print("     Hidden :", ht.detach().numpy())

    if t > 0:
        prev_h = out_man[t - 1]
    else:
        prev_h = torch.zeros((ht.shape))
    ot = ht + torch.matmul(prev_h, torch.transpose(w_hh, 0, 1)) + b_hh
    ot = torch.tanh(ot)
    out_man.append(ot)

    print("     Output (manual) :", ot.detach().numpy())
    print("     RNN output      :", output[:, t].detach().numpy())
    print()

Time step 0 =>        
     Input :  [[1. 1. 1. 1. 1.]]
     Hidden : [[-0.3161478   0.64722455]]
     Output (manual) : [[-0.21046415  0.56788784]]
     RNN output      : [[-0.3519801   0.52525216]]

Time step 1 =>        
     Input :  [[2. 2. 2. 2. 2.]]
     Hidden : [[-0.73478645  1.2972739 ]]
     Output (manual) : [[-0.5741978  0.7945334]]
     RNN output      : [[-0.68424344  0.76074266]]

Time step 2 =>        
     Input :  [[3. 3. 3. 3. 3.]]
     Hidden : [[-1.153425   1.9473233]]
     Output (manual) : [[-0.8130059  0.918174 ]]
     RNN output      : [[-0.8649416  0.9046636]]



---
### Implementing RNNs for sequence modeling in PyTorch

In [104]:
from torchtext.datasets import IMDB

train_dataset = IMDB(split="train", root="./datasets")
test_dataset = IMDB(split="test", root="./datasets")

In [105]:
from torch.utils.data.dataset import random_split

train_dataset, valid_dataset = random_split(list(train_dataset), [20000, 5000])

In [106]:
import re
from collections import Counter, OrderedDict


def tokenizer(text: str):
    text = re.sub("<[^>]*>", "", text)
    emoticons = re.findall("(?::|;|=)(?:-)?(?:\)|\(|D|P)", text.lower())
    text = re.sub("[\W]+", " ", text.lower()) + " ".join(emoticons).replace("-", "")
    tokenized = text.split()
    return tokenized

<>:7: SyntaxWarning: invalid escape sequence '\)'
<>:8: SyntaxWarning: invalid escape sequence '\W'
<>:7: SyntaxWarning: invalid escape sequence '\)'
<>:8: SyntaxWarning: invalid escape sequence '\W'
/tmp/ipykernel_190259/4095290228.py:7: SyntaxWarning: invalid escape sequence '\)'
  emoticons = re.findall("(?::|;|=)(?:-)?(?:\)|\(|D|P)", text.lower())
/tmp/ipykernel_190259/4095290228.py:8: SyntaxWarning: invalid escape sequence '\W'
  text = re.sub("[\W]+", " ", text.lower()) + " ".join(emoticons).replace("-", "")


In [107]:
token_counts = Counter()

for label, line in train_dataset:
    tokens = tokenizer(line)
    token_counts.update(tokens)

print("Vocab-size: ", len(token_counts))

Vocab-size:  69252


In [108]:
# Step 3: Encoding each unique token into integers
from torchtext.vocab import vocab

sorted_by_freq_tuples = sorted(token_counts.items(), key=lambda x: x[1], reverse=True)

ordered_dict = OrderedDict(sorted_by_freq_tuples)
vocab = vocab(ordered_dict)
#  we will prepend two special tokens to the vocabulary – the padding and the unknown token:
vocab.insert_token("<pad>", 0)
vocab.insert_token("<unk>", 1)
vocab.set_default_index(1)

In [109]:
print(
    [
        vocab[token]
        for token in ["this", "is", "a", "working", "example", "rerwrew3", " "]
    ]
)

[11, 7, 4, 788, 468, 1, 1]


In [110]:
## Step 3A: Define the functions for transformations

text_pipeline = lambda x: [vocab[token] for token in tokenizer(x)]  # noqa: E731
label_pipleline = lambda x: 1.0 if x == 2 else 0.0  # noqa: E731

In [111]:
## Step 3-B: Wrap the encode and transformation function


def collate_batch(batch):
    label_list, text_list, lengths = [], [], []

    for _label, _text in batch:
        label_list.append(label_pipleline(_label))
        processed_text = torch.tensor(text_pipeline(_text), dtype=torch.int64)
        text_list.append(processed_text)
        lengths.append(processed_text.size(0))

    label_list = torch.tensor(label_list)
    lengths = torch.tensor(lengths)
    padded_text_list = nn.utils.rnn.pad_sequence(text_list, batch_first=True)

    return padded_text_list, label_list, lengths

In [112]:
## Taking a small batch
from torch.utils.data import DataLoader

dataloader = DataLoader(
    train_dataset, batch_size=4, shuffle=False, collate_fn=collate_batch
)

text_batch, label_batch, length_batch = next(iter(dataloader))

print(text_batch[1].shape)
print(label_batch)
print(length_batch)

torch.Size([171])
tensor([1., 0., 1., 1.])
tensor([132, 161, 131, 171])


In [113]:
BATCH_SIZE = 32
train_dl = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch
)
valid_dl = DataLoader(
    valid_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch
)
test_dl = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch
)

-  The output will have the dimensionality `batchsize × input_length × embedding_dim`, where `embedding_dim` is the size of the embedding features (here, set to 3). 

In [114]:
embedding = nn.Embedding(num_embeddings=10, embedding_dim=3, padding_idx=0)

# example
text_encoded_input = torch.LongTensor([[1, 2, 4, 5, 6, 1], [4, 3, 2, 0, 6, 1]])
print(embedding(text_encoded_input))

tensor([[[-1.3871,  0.5451,  0.0388],
         [-1.1327,  0.1908, -0.1563],
         [-0.4916, -1.0647, -0.3736],
         [ 0.5880, -1.4682, -0.0671],
         [ 1.9113,  0.3674,  1.4034],
         [-1.3871,  0.5451,  0.0388]],

        [[-0.4916, -1.0647, -0.3736],
         [ 0.9941, -1.6459,  0.1540],
         [-1.1327,  0.1908, -0.1563],
         [ 0.0000,  0.0000,  0.0000],
         [ 1.9113,  0.3674,  1.4034],
         [-1.3871,  0.5451,  0.0388]]], grad_fn=<EmbeddingBackward0>)


---
## Building an RNN Model

In [115]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size) -> None:
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers=2, batch_first=True)
        # Can use nn.GRU / nn.LSTM too
        self.fc = nn.Linear(hidden_size, 1)  # fc = Fully Connected

    def forward(self, x):
        _, hidden = self.rnn(x)
        out = hidden[-1, :, :]

        out = self.fc(out)
        return out

In [116]:
model = RNN(64, 32)
print(model)
model(torch.randn(5, 3, 64))

RNN(
  (rnn): RNN(64, 32, num_layers=2, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)


tensor([[ 0.0246],
        [-0.0826],
        [ 0.2085],
        [ 0.3292],
        [-0.0565]], grad_fn=<AddmmBackward0>)

---
# Building an RNN model for the sentiment analysis task

In [117]:
device = "cuda"

In [118]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, rnn_hidden_size, fc_hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.LSTM(embed_dim, rnn_hidden_size, batch_first=True)
        self.fc1 = nn.Linear(rnn_hidden_size, fc_hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(fc_hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, text, lengths):
        out = self.embedding(text)
        out = nn.utils.rnn.pack_padded_sequence(
            out, lengths.cpu().numpy(), enforce_sorted=False, batch_first=True
        )
        out, (hidden, cell) = self.rnn(out)
        out = hidden[-1, :, :]

        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.sigmoid(out)

        return out

In [119]:
vocab_size = len(vocab)
embed_dim = 20
rnn_hidden_size = 64
fc_hidden_size = 64

# torch.manual_seed(1)
model = RNN(vocab_size, embed_dim, rnn_hidden_size, fc_hidden_size).to(device)
model

RNN(
  (embedding): Embedding(69254, 20, padding_idx=0)
  (rnn): LSTM(20, 64, batch_first=True)
  (fc1): Linear(in_features=64, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [120]:
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [121]:
def train(dataloader: DataLoader):
    model.train()
    total_acc, total_loss = 0, 0
    for text_batch, label_batch, lengths in dataloader:
        text_batch, label_batch = text_batch.to(device), label_batch.to(device)
        optimizer.zero_grad()
        pred = model(text_batch, lengths)[:, 0]
        loss = loss_fn(pred, label_batch.float())
        loss.backward()
        optimizer.step()

        total_acc += ((pred >= 0.5).float() == label_batch).float().sum().item()
        total_loss += loss.item() * label_batch.size(0)

    return total_acc / len(dataloader.dataset), total_loss / len(dataloader.dataset)  # type: ignore

In [122]:
def evaluate(dataloader: DataLoader):
    model.eval()
    total_acc, total_loss = 0, 0
    with torch.no_grad():
        for text_batch, label_batch, lengths in dataloader:
            text_batch, label_batch = text_batch.to(device), label_batch.to(device)
            pred = model(text_batch, lengths)[:, 0]
            loss = loss_fn(pred, label_batch.float())
            total_acc += ((pred >= 0.5).float() == label_batch).sum().item()
            total_loss += loss.item() * label_batch.size(0)

    return total_acc / len(dataloader.dataset), total_loss / len(dataloader.dataset)  # type: ignore

In [123]:
if torch.cuda.is_available():
    print(f"GPU is available: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not found. Training will run on CPU.")

GPU is available: NVIDIA GeForce RTX 3050


In [124]:
NUM_EPOCHS = 10
# torch.manual_seed(1)

for e in range(NUM_EPOCHS):
    acc_train, loss_train = train(train_dl)
    acc_valid, loss_valid = evaluate(valid_dl)
    print(f"Epoch {e} accuracy : {acc_train:.4f} | val_accuracy: {acc_valid:.4f}")

Epoch 0 accuracy : 0.6051 | val_accuracy: 0.6978
Epoch 1 accuracy : 0.6854 | val_accuracy: 0.7138
Epoch 2 accuracy : 0.7721 | val_accuracy: 0.7852
Epoch 3 accuracy : 0.8284 | val_accuracy: 0.8080
Epoch 4 accuracy : 0.8675 | val_accuracy: 0.8236
Epoch 5 accuracy : 0.8862 | val_accuracy: 0.8414
Epoch 6 accuracy : 0.9125 | val_accuracy: 0.8498
Epoch 7 accuracy : 0.9306 | val_accuracy: 0.8472
Epoch 8 accuracy : 0.9377 | val_accuracy: 0.8456
Epoch 9 accuracy : 0.9525 | val_accuracy: 0.8536


In [125]:
acc_test, _ = evaluate(test_dl)
print(f"Test Accuracy: {acc_test:.4f}")

Test Accuracy: 0.9636


In [126]:
def predict_sentiment(text, model, vocab, device):
    model.eval()
    with torch.no_grad():
        # 1. Tokenize (using the same tokenizer used for training)
        tokens = tokenizer(text)

        # 2. Convert tokens to integers using your vocabulary
        # If a word isn't in vocab, use the <unk> index (usually 0 or 1)
        indices = [
            vocab[token] if token in vocab else vocab["<unk>"] for token in tokens
        ]

        # 3. Create tensors and move to device
        text_tensor = (
            torch.tensor(indices, dtype=torch.long).unsqueeze(0).to(device)
        )  # Batch size 1
        lengths = torch.tensor([len(indices)])  # Length of this specific review

        # 4. Predict
        prediction = model(text_tensor, lengths)[:, 0]

        # 5. Interpret result
        label = 1 if prediction.item() >= 0.5 else 0
        sentiment = "Positive" if label == 1 else "Negative"

        print(f"Review: {text}")
        print(f"Prediction: {sentiment} ({prediction.item():.4f})")
        return label

In [129]:
my_review = "This movie was absolutely fantastic! Great acting and plot."
predict_sentiment(my_review, model, vocab, device)
print()
bad_review = "I liked the movie but the main actor was insufferable. Would not recommend if you like deep character arcs!"
predict_sentiment(bad_review, model, vocab, device)
print()

Review: This movie was absolutely fantastic! Great acting and plot.
Prediction: Positive (0.8715)

Review: I liked the movie but the main actor was insufferable. Would not recommend if you like deep character arcs!
Prediction: Negative (0.3316)

